# Lab06 · Mi primer modelo
**¿Puede un árbol reconocer paltas maduras?** — INF670 · Introducción a Ciencia de Datos e IA · UATF

1. Escribe tu identificador en `ESTUDIANTE` (celda de inicio).
2. Ejecuta las celdas de código **en orden** con `Shift+Enter`.
3. Responde en las celdas ✍️ (doble clic para editar).
4. No cambies la semilla, la partición ni `max_depth` para conseguir una cifra más alta.

> Datos sintéticos, creados con fines didácticos.

## Celda de inicio

In [ ]:
from pathlib import Path
import sys, hashlib
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

ESTUDIANTE = "adalid@portillo"   # Escribe aqui tu identificador.
SEMILLA = int(hashlib.sha256(ESTUDIANTE.encode()).hexdigest(), 16) % 1000
BASE = Path.cwd()
ARCHIVO = BASE / "planilla_paltas_revisada.csv"
SALIDAS = BASE / "salidas"
SALIDAS.mkdir(exist_ok=True)
HUELLA_INICIAL = hashlib.sha256(ARCHIVO.read_bytes()).hexdigest()

print("Estudiante:", ESTUDIANTE, "| Semilla:", SEMILLA)
print("Entorno virtual activo:", sys.prefix != sys.base_prefix)
print("Planilla encontrada:", ARCHIVO.exists())

## Tarea 1 · Preparar: pistas, respuesta y separación

In [ ]:
df = pd.read_csv(ARCHIVO, encoding="utf-8")
print("Filas y columnas:", df.shape)
print("Datos faltantes:", int(df.isna().sum().sum()),
      "| Filas repetidas:", int(df.duplicated().sum()))

X = df[["firmeza", "dias_cosecha", "peso_g"]]   # pistas
y = df["madura"]                                # respuesta
display(X.head())
print(y.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEMILLA, stratify=y
)
print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)
print("Paltas en ambos conjuntos:",
      len(set(X_train.index) & set(X_test.index)))

### ✍️ Respuesta 1

(a) ¿Qué contiene X y qué contiene y? (b) ¿Por qué `id` y `madura` no están dentro de X? (c) ¿Cuántas paltas quedaron para entrenar y cuántas para la prueba? (d) ¿Por qué «Paltas en ambos conjuntos» debe ser 0?

*(a): `x` contiene las columnas de entrada/caracteristicas; `y` contiene la columna objetivo/varieable_Dependiente*

*(b): `id` es propio del formulario; `madura` es lo que se va predecir*

*(c): 80-20%, 96-24*

*(d): porque los datos de entrenamiento no deben estar en prueba a la misma vez, con `len(>0)` entenderiamos que hubo una interseccion de indices por `set()&set()`, por lo tanto el modelo no seria confiable con sus estadisticos de sus pruebas*

## Tarea 2 · Entrenar y leer el árbol

In [ ]:
modelo = DecisionTreeClassifier(max_depth=1, random_state=42)
modelo.fit(X_train, y_train)
print(export_text(modelo, feature_names=list(X.columns)))

### ✍️ Respuesta 2

(a) Escribe con tus palabras la pregunta que aprendió tu árbol y qué responde en cada rama. (b) ¿Quién eligió ese umbral: tú o el entrenamiento? (c) Compáralo con el valor de firmeza que encontraste a ojo en el Lab05.

*(a): aprendio a cladificar entre las 3 variables independientes, una sola que clasifica mejor entre maduras o no; responde que cuando firmeza es <= 5.50 la palta no es madura, de lo contrario si lo es*

*(b): la clase `DecicisionTreeClassifier`*

*(c): '5 aprox'; '5,5' --> 0.5 diff::: me acerce al valor*

## Tarea 3 · Evaluar con paltas que el modelo no vio

In [ ]:
pred = modelo.predict(X_test)
resultado = df.loc[X_test.index, ["id", "firmeza", "dias_cosecha", "peso_g"]].copy()
resultado["real"] = y_test
resultado["predicha"] = pred
resultado["acierto"] = resultado["real"] == resultado["predicha"]

aciertos = int(resultado["acierto"].sum())
total = len(resultado)
frecuente = y_train.mode()[0]
print(f"Aciertos: {aciertos}/{total} | Exactitud: {aciertos / total:.1%}")
print(f"Referencia (responder siempre '{frecuente}'): "
      f"{(y_test == frecuente).mean():.1%}")
display(pd.crosstab(resultado["real"], resultado["predicha"],
                    rownames=["Real"], colnames=["Predicha"]))
print("Paltas en las que el modelo se equivoco:")
display(resultado[~resultado["acierto"]])
resultado.to_csv(SALIDAS / "predicciones_prueba.csv", index=False)

In [ ]:
plt.figure(figsize=(6, 4))
for ok, color, nombre in [(True, "tab:green", "Acierto"), (False, "tab:red", "Error")]:
    grupo = resultado[resultado["acierto"] == ok]
    plt.scatter(grupo["firmeza"], grupo["dias_cosecha"],
                c=color, s=70, alpha=0.6, label=nombre)
plt.title(f"{ESTUDIANTE} | Aciertos y errores en la prueba")
plt.xlabel("Firmeza (1 = muy dura, 10 = muy blanda)")
plt.ylabel("Dias desde la cosecha")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(SALIDAS / "aciertos_y_errores.png", dpi=150)
plt.show()

### ✍️ Respuesta 3

(a) Escribe tu resultado como fracción (aciertos/total) y compáralo con la referencia. (b) Elige un error de tu tabla: anota su `id`, qué era en realidad y qué predijo el modelo. Si no hubo errores, explica por qué eso no demuestra que el modelo sea perfecto. (c) Para la tienda, ¿qué error cuesta más: vender como madura una palta que no lo está, o apartar una que sí lo estaba? ¿Cuántos hubo de cada tipo?

*(a): 21/24 --> >75%; no dio No para el 50% de los datos*

*(b): P100 = Predicha:No; Acierto:False /////en realidad era `Si` ;;;HUBO ERROR. en firmeza esta con 3, lo que indica que la palta esta muy dura, por lo tanto deberia ser inmadura, tal cual dijo el modelo*

*(c): Le cuesta mas apartar una que si lo estaba, porque primero no vende y por lo tanto no gana, segundo como lo aparta estando ya madura, sigue madurando pero madura tanto que se excede y ya no es comestible por lo tanto pierde. /Resumen: pierde. ||hubo dos inmaduras pero los clasifico como maduras y una madura que lo clasifico como inmadura;;dije que era error*

## Tarea 4 · ¿Memorizar o generalizar?

In [ ]:
grande = DecisionTreeClassifier(random_state=42)   # sin limite de preguntas
grande.fit(X_train, y_train)
for nombre, m in [("Arbol simple", modelo), ("Arbol grande", grande)]:
    print(f"{nombre}: {m.get_depth()} nivel(es) de preguntas"
          f" | entrenamiento {m.score(X_train, y_train):.1%}"
          f" | prueba {m.score(X_test, y_test):.1%}")

### ✍️ Respuesta 4

(a) ¿Qué árbol acierta más con las paltas de entrenamiento? ¿Y con las de prueba? (b) ¿Cuál de los dos elegirías para la tienda y por qué? (c) Explica con tus palabras la diferencia entre memorizar y generalizar.

*(a):  el arbol grande con train; arbol simple con tests*

*(b): elijo el simple porque predice mejor ocn datos de prueba y tiene menos riesgo de equivocarse al clasificar otras plantas*

*(c): menmoriza entrada y salida; aprender patrones que luego con eso podremos elavorar salidas correctas con entradas nuevas*

## Tarea 5 · Consultar una palta nueva

In [ ]:
palta_nueva = pd.DataFrame([[7, 9, 230]], columns=X.columns)
display(palta_nueva)
print("Prediccion:", modelo.predict(palta_nueva)[0])

In [ ]:
# Reto: crea tu propia palta con valores permitidos por el diccionario
# (firmeza 1-10, dias_cosecha 0-30, peso_g 100-500) y pide la prediccion.
# Usa como modelo la celda anterior.

In [ ]:
palta_loo = pd.DataFrame([[5, 2, 100]], columns=X.columns)
display(palta_loo)
print('prediccion:', modelo.predict(palta_nueva)[0])

### ✍️ Respuesta 5

(a) ¿Qué valores elegiste y qué predijo el modelo? (b) ¿Puedes afirmar que esa predicción es un acierto? ¿Qué te faltaría para saberlo?

*(a): 5 porque esta entre madura y no madura, 2 justo por lo anterior, se cosecho hace dos dias, 100 porque no pesa tanto. PREDIJO SI, tiene sentido*

*(b): si, por lo anterior; tener la palta real*

## Tarea 6 · Contraste: agrupar sin respuestas

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

medidas = df[["peso_g", "firmeza"]]
Z = StandardScaler().fit_transform(medidas)   # misma escala para ambas medidas
grupos = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(Z)

plt.figure(figsize=(6, 4))
for g in (0, 1):
    plt.scatter(medidas.loc[grupos == g, "peso_g"],
                medidas.loc[grupos == g, "firmeza"],
                s=60, alpha=0.6, label=f"Grupo {g}")
plt.title(f"{ESTUDIANTE} | Grupos sin etiquetas")
plt.xlabel("Peso (g)")
plt.ylabel("Firmeza (1 = muy dura, 10 = muy blanda)")
plt.legend()
plt.tight_layout()
plt.savefig(SALIDAS / "grupos_sin_etiquetas.png", dpi=150)
plt.show()
print("Grupos frente a la VARIEDAD:")
print(pd.crosstab(grupos, df["variedad"], rownames=["Grupo"]))
print("Grupos frente a la MADUREZ:")
print(pd.crosstab(grupos, df["madura"], rownames=["Grupo"]))

### ✍️ Respuesta 6

(a) ¿Qué recibió el árbol que el agrupador no recibió? (b) Según la tabla, ¿a qué se parecen los grupos: a la madurez o a la variedad? (c) ¿Por qué no puedes llamar «Hass» al grupo 0 solo por su número?

*(a): recibio respuestas correctas de `y_train`, pero el agrupador no recibio etiquetas solo las medidas de `peso` y `firmeza`*

*(b): se parece mas a la `variedad`*

*(c): porque los numeros son asignados automaticamente por `kmeans`, en este caso vemos que 0 corresponde a `hass` pero no siempre sera asi*

## Tarea 7 · Dictamen

### ✍️ Dictamen

¿La tienda puede **USAR** el árbol simple, **USARLO CON SUPERVISIÓN** o **NO USARLO TODAVÍA** para separar paltas maduras? Justifica en 80-120 palabras citando tu fracción de aciertos, la referencia y al menos un error con su `id`.

*Usarlo con Supervicion, porque el arbol simple obtuvo un porcentaje alto de 87% con 21/24 de aciertos, por lo que decimos que clasifica bien habiendo tenido una referencia del 50%. El error estaba en P100 por la palta que no tenia sentido que tenga madurez con esos datos registrados. Ademas que con las pruebas totalmente externas que ingrese si dieron como resultado una respuesta correcta esperada.*

## Evidencia final
Ejecuta esta celda al terminar y toma la captura de pantalla que pide la guía.

In [ ]:
import platform, datetime
assert ESTUDIANTE != "ID_ASIGNADO", "Escribe tu identificador en la celda de inicio."
huella_final = hashlib.sha256(ARCHIVO.read_bytes()).hexdigest()
clave = f"{ESTUDIANTE}{huella_final[:12]}{aciertos}/{total}"
codigo = hashlib.sha256(clave.encode()).hexdigest()[:8].upper()
print("Estudiante:", ESTUDIANTE, "| Semilla:", SEMILLA)
print("Equipo (VM):", platform.node())
print("Fecha y hora:", datetime.datetime.now().strftime("%Y-%m-%d %H:%M"))
print("Planilla sin modificar:", huella_final == HUELLA_INICIAL)
print("Aciertos en la prueba:", f"{aciertos}/{total}")
print("Codigo de verificacion:", codigo)
print("Archivos en salidas:", sorted(p.name for p in SALIDAS.iterdir()))